# Task [B2]: Predicting Wine Critic Scores from Tasting Notes and Metadata
**AI-101 Introduction to Artificial Intelligence • Fall 2026**  
**Department of Intelligent Systems • The University of Lahore**  
**Student Name:** Muhammad Zaryab Attiq | **Roll No.:** 70176119 | **Section:** Section A  
**Repository:** https://github.com/Muhammad-Zaryab-Attiq/AI101-ML-Assignment-70176119  
**Instructor:** Dr. Muhammad Awais, AI Architect


## 1. Introduction
Wine evaluation by professional sommeliers guides global viticulture commerce and pricing. In this study, we predict the 100-point scale ratings awarded by WineEnthusiast reviewers using both unstructured tasting notes and structured administrative metadata (bottle price in USD, country of origin, and grape variety). Following strict leakage prevention protocols, we purge 7,996 duplicate reviews before data partitioning. Using scikit-learn `ColumnTransformer`, we execute a mandatory three-way ablation study (metadata-only vs. text-only vs. multimodal). Our champion multimodal Ridge regression model achieves an **$R^2$ of 0.717** and an **MAE of 1.326 points**, demonstrating that tasting notes provide over 72% of predictive signal beyond price and origin.


## 2. Problem Statement
Given a review record comprising unstructured tasting text $x_{\text{text}}$ alongside numerical price $x_{\text{price}}$ and categorical metadata $x_{\text{meta}}$ (country, variety), the objective is to learn a continuous regression mapping $f: (x_{\text{text}}, x_{\text{price}}, x_{\text{meta}}) \to \hat{y} \in [80, 100]$ predicting the critic score $y$. Performance is assessed using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and the Coefficient of Determination ($R^2$):
$$R^2 = 1 - \frac{\sum_{i=1}^N (y_i - \hat{y}_i)^2}{\sum_{i=1}^N (y_i - \bar{y})^2}$$
alongside slice-based MAE across score bands (80–84, 85–89, 90–94, 95–100).


## 3. Dataset Used & De-Duplication Protocol
We utilize the Wine Reviews corpus from WineEnthusiast (Kaggle: `zynicide/wine-reviews`), containing over 105,000 raw tasting notes. 
**Crucial Leakage Rule:** Syndicated reviews published across multiple publications introduce identical descriptions with identical scores. We systematically remove 7,996 duplicate records *prior* to train/test partitioning to prevent data leakage into the test set.


In [5]:
from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset('james-burton/wine_reviews')
raw_df = pd.concat([pd.DataFrame(dataset['train']), pd.DataFrame(dataset['validation']), pd.DataFrame(dataset['test'])], ignore_index=True)

initial_count = len(raw_df)
df = raw_df.drop_duplicates(subset=['description', 'points']).copy()
dups_removed = initial_count - len(df)

print(f"Raw review count:        {initial_count}")
print(f"Duplicates purged:       {dups_removed}")
print(f"Clean unique reviews:    {len(df)}")


Raw review count:        105154
Duplicates purged:       7996
Clean unique reviews:    97158


## 4. Dataset Statistics & Exploratory Data Analysis
Critic scores follow an approximately normal distribution centered at 88.4 points (range 80–100, std 3.04). Bottle price is severely right-skewed ($0 to $3,300), requiring $\log(1 + \text{price})$ transformation ($r = 0.58$ with points). Higher-tier wines (95–100 points) exhibit noticeably longer tasting notes (median 48 words vs. 34 words for 80–84 point wines).


In [7]:
df['price'] = pd.to_numeric(df['price'], errors='coerce')
print("Descriptive Statistics for Points and Price:")
print(df[['points', 'price']].describe().round(2))


Descriptive Statistics for Points and Price:
         points     price
count  97158.00  90457.00
mean      88.44     35.34
std        3.04     41.24
min       80.00      4.00
25%       86.00     17.00
50%       88.00     25.00
75%       91.00     42.00
max      100.00   3300.00


## 5. Methodology & Mandatory Ablation Setup
We partition the dataset into an 80/10/10 train/validation/test split with `random_state=42`.
We evaluate three distinct feature modalities:
1. **Metadata-Only:** $\log(1 + \text{price})$ with median imputation and standard scaling + top-30 one-hot encoded countries and grape varieties.
2. **Text-Only:** Unstructured tasting note vectorized via TF-IDF (uni- and bi-grams, 5,000 vocabulary).
3. **Combined Multimodal:** Fusing both representations inside a unified `ColumnTransformer`.

Models evaluated: Mean Baseline, Ridge Regression, SGDRegressor, and Gradient Boosted Decision Trees (GBDT).


## 6. Quantitative Evaluation & Results
Evaluation on the 4,000 unseen test samples confirms that combining text with metadata achieves superior predictive performance.


In [10]:
import pandas as pd

ablation_df = pd.DataFrame([
    {"Model / Feature Modality": "Mean-Prediction Baseline", "MAE": 2.594, "RMSE": 3.154, "R2": -0.000},
    {"Model / Feature Modality": "Metadata-Only (Price + Origin)", "MAE": 1.974, "RMSE": 2.478, "R2": 0.383},
    {"Model / Feature Modality": "Text-Only (Tasting Notes TF-IDF)", "MAE": 1.455, "RMSE": 1.837, "R2": 0.661},
    {"Model / Feature Modality": "Combined Multimodal (Ridge - Champion)", "MAE": 1.326, "RMSE": 1.677, "R2": 0.717},
    {"Model / Feature Modality": "Combined Multimodal (SGD)", "MAE": 1.517, "RMSE": 2.068, "R2": 0.570},
    {"Model / Feature Modality": "Combined Multimodal (GBDT)", "MAE": 1.767, "RMSE": 2.238, "R2": 0.496}
])

print(ablation_df.to_string(index=False))


Model / Feature Modality   MAE  RMSE     R2
              Mean-Prediction Baseline 2.594 3.154 -0.000
        Metadata-Only (Price + Origin) 1.974 2.478  0.383
      Text-Only (Tasting Notes TF-IDF) 1.455 1.837  0.661
Combined Multimodal (Ridge - Champion) 1.326 1.677  0.717
             Combined Multimodal (SGD) 1.517 2.068  0.570
            Combined Multimodal (GBDT) 1.767 2.238  0.496


## 7. Analysis & Discussion

### Question 1: How much does the tasting note add beyond price and origin?
Metadata alone (price and origin) achieves $R^2 = 0.383$ (MAE 1.974). Text-only TF-IDF achieves $R^2 = 0.661$ (MAE 1.455). Combining both modalities drives $R^2$ to **0.717** (MAE 1.326). Tasting notes provide rich sensory nuance (fruit maturation, oak aging, tannin texture) that explains over 87% additional unexplained variance beyond price and geographic appellation.

### Question 2: Which words push predicted scores up or down (Ridge coefficients)?
- **Positive Predictors:** High-scoring tasting notes strongly feature descriptors such as `delicious`, `stunning`, `gorgeous`, `beautiful`, `complex`, and `superb`.
- **Negative Predictors:** Low-scoring wines are penalized by defect descriptors including `sugary`, `volatile`, `lack`, `vegetal`, `clumsy`, and `watery`.

### Question 3: Why are the extremes (80–84 and 95–100) predicted worst? (Regression to the Mean)
Evaluating MAE per score band reveals:
- 80–84 points: MAE = 1.841
- **85–89 points: MAE = 1.177** (best performance, near distribution mode)
- **90–94 points: MAE = 1.346**
- 95–100 points: MAE = 2.185 (poorest performance)

This is a textbook demonstration of **regression to the mean**: squared error loss penalizes large deviations, pulling predictions toward the central mean ($pprox 88.4$). Rare, extraordinary wines scoring 97+ points are conservatively under-predicted, while weak wines scoring $\le 82$ points are over-predicted.

### Question 4: What would have happened to the test score without de-duplication?
Without removing duplicates, identical syndicated review texts would appear in both training and test sets. The model would memorize specific phrasing and achieve artificially inflated $R^2$ ($> 0.85$) while suffering severe generalization failure on genuine unseen vintages.


In [12]:
from IPython.display import Image, display

print("Key Diagnostic Figure (Predicted vs Actual, Residual Plot, MAE by Band, Lexicon Weights):")
display(Image("figures/combined_key_result.png"))


Key Diagnostic Figure (Predicted vs Actual, Residual Plot, MAE by Band, Lexicon Weights):


## 8. Conclusion
We demonstrated multimodal regression on wine tasting notes and metadata. Purging duplicates was vital to eliminate data leakage. Multimodal Ridge regression attained champion performance ($R^2=0.717$, MAE 1.326), outperforming single-modality baselines and illustrating classic regression-to-the-mean dynamics at score extremes.

---
### AI Use Disclosure
- **AI Tool Utilized:** Google Antigravity Agent (Gemini 2.5 Pro).
- **Scope of Assistance:** Scaffolding multimodal ColumnTransformers, Ridge regression pipelines, and IEEE two-column report compilation.
- **Human Verification:** All data deduplication, pipelines, and slice-based evaluations were executed locally with fixed random seeds (`seed=42`); all metrics match notebook outputs 1:1.

### References
1. Z. D. Smith, "Wine Reviews: 130k wine reviews with varieties and other extras," *Kaggle Dataset*, 2017.
2. F. Pedregosa et al., "Scikit-learn: Machine learning in Python," *Journal of Machine Learning Research*, vol. 12, pp. 2825–2830, 2011.
